In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [20]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer

In [2]:
#Load data
df = pd.read_csv("../data/mental_health_data.csv")
df.head()

,Name,Age,Marital_Status,Education_Level,Number_of_Children,Smoking_Status,Physical_Activity_Level,Employment_Status,Income,Alcohol_Consumption,Dietary_Habits,Sleep_Patterns,History_of_Mental_Illness,History_of_Substance_Abuse,Family_History_of_Depression,Chronic_Medical_Conditions,Record_ID,data_split
0,Christine Barker,31,Married,Bachelor's Degree,2,Non-smoker,Active,Unemployed,26265.67,Moderate,Moderate,Fair,Yes,No,Yes,Yes,100000,validation
1,Jacqueline Lewis,55,Married,High School,1,Non-smoker,Sedentary,Employed,42710.36,High,Unhealthy,Fair,Yes,No,No,Yes,100001,validation
2,Shannon Church,78,Widowed,Master's Degree,1,Non-smoker,Sedentary,Employed,125332.79,Low,Unhealthy,Good,No,No,Yes,No,100002,validation
3,Charles Jordan,58,Divorced,Master's Degree,3,Non-smoker,Moderate,Unemployed,9992.78,Moderate,Moderate,Poor,No,No,No,No,100003,validation
4,Michael Rich,18,Single,High School,0,Non-smoker,Sedentary,Unemployed,8595.08,Low,Moderate,Fair,Yes,No,Yes,Yes,100004,validation


From the EDA, we had a general understanding  of the data, including the distribution of features, the relationship between features and the target variable. In this scripts, I'll start testing the model and some hypeterparameter tuning. 

## Transformation

In [5]:
exl_cols = ["Name", "Record_ID", "data_split", "History_of_Mental_Illness"]

In [12]:
df["Number_of_Children"] = df["Number_of_Children"].replace(4, 3)

I would not group widowed and divorced together in this case. Normally, considering the different use cases and how the results will be used, I would apply some consideration to treating vulnerable customers. If the results could result in treating customers unfairly, then I would group vulnerable customers into another set. However, in this specific situation, I will not take any action, as there is an important link between marital status and the likelihood of depression.

I will treat PHD as a distinct education level, as merging it with Master's would remove information from the model. If education is associated with differences in the target, combining PhD with Master's could hide that difference.

In [13]:
train = df[df["data_split"] == "train"].copy()
validation = df[df["data_split"] == "validation"].copy()
test = df[df["data_split"] == "test"].copy()

X_train = train.drop(columns=exl_cols)
y_train = train["History_of_Mental_Illness"]

X_val = validation.drop(columns=exl_cols)
y_val = validation["History_of_Mental_Illness"]

X_test = test.drop(columns=exl_cols)
y_test = test["History_of_Mental_Illness"]

In [15]:
numeric_features = ["Age", "Income"]
categorical_features = X_train.select_dtypes(
    include=["object"]
).columns.tolist()
categorical_features.append("Number_of_Children")
categorical_features

/var/folders/wh/mcjq580j09j41w3vqk9sxwd80000gn/T/ipykernel_26357/3064991036.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X_train.select_dtypes(


['Marital_Status',
 'Education_Level',
 'Smoking_Status',
 'Physical_Activity_Level',
 'Employment_Status',
 'Alcohol_Consumption',
 'Dietary_Habits',
 'Sleep_Patterns',
 'History_of_Substance_Abuse',
 'Family_History_of_Depression',
 'Chronic_Medical_Conditions',
 'Number_of_Children']

In [16]:
#Frequent encoding for categorical features
marital_frequency = (X_train["Marital_Status"].value_counts(normalize=True))
X_train["Marital_Status"] = (X_train["Marital_Status"].map(marital_frequency).fillna(0))
X_val["Marital_Status"] = (X_val["Marital_Status"].map(marital_frequency).fillna(0))
X_test["Marital_Status"] = (X_test["Marital_Status"].map(marital_frequency).fillna(0))

In [17]:
#Orindial encoding 
ordinal_features = ["Education_Level", "Physical_Activity_Level", "Sleep_Pattern","Alchol_Consumption","Dietary_Habits", "Number_of_Children"]

In [18]:
ordinal_categories = [
    [   "High School",
        "Associate Degree",
        "Bachelor's Degree",
        "Master's Degree",
        "PhD"
    ],

    [   "Sedentary",
        "Moderate",
        "Active"
    ],

    [   "Poor",
        "Fair",
        "Good"
    ],

    [   "Low",
        "Moderate",
        "High"
    ],

    [
        "Unhealthy",
        "Moderate",
        "Healthy"
    ],

    [
        "0",
        "1",
        "2",
        "3"
    ]
]

In [21]:
ordinal_encoder = OrdinalEncoder(
    categories=ordinal_categories,
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

In [22]:
#Onehot encoding
onehot_features = [
    "Smoking_Status",
    "Employment_Status",
    "History_of_Substance_Abuse",
    "Family_History_of_Depression",
    "Chronic_Medical_Conditions"
]

In [23]:
onehot_encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=True
)

In [ ]:
#Numeric imputer
numeric_features

['Age', 'Income']

From EDA, we noticed that Income is a right-skewed feature, so normally we could take some mathmatical transformation to it, like Log(x) to reduce the effect of the long right tail.  However, using the original Income values makes the model output much easier explain to stakeholders than log(x).Particularly when using a more complex model such as XGBoost. Therefore, I'd like retain Income in its original scale and apply a cap to limit the influence of extreme values.

In [ ]:
df["Income"] = df["Income"].clip(upper=150000)
X_train["Income"] = X_train["Income"].clip(upper=150000)
X_val["Income"] = X_val["Income"].clip(upper=150000)
X_test["Income"] = X_test["Income"].clip(upper=150000)